# Railway Headway & Capacity Simulator
## Google Colab Application Launcher

**Software Version:** `0.1.0-dev`  
**Current Phase:** `P00 — Project Foundation`  
**Interface:** Gradio Blocks  
**Execution Mode:** Google Colab (Inline Display)

---

### Instructions for Running in Google Colab:
1. **Step 1:** Verify runtime compatibility (Python >= 3.10 required).
2. **Step 2:** Ensure the `railway-headway-simulator` repository is present and installed.
3. **Step 3:** Initialize runtime directories and logging.
4. **Step 4:** Launch the interactive Gradio interface directly in the cell below.

*Note: Per Governance Directive RHS-MASTER-001, all engineering calculations execute in the Python package in the background. End users do not need to edit code.*


In [1]:
# Step 1: Verify Python Runtime Compatibility
import sys
import platform

print("=== Google Colab Runtime Verification ===")
print(f"Python Version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")

if sys.version_info < (3, 10):
    raise RuntimeError(
        f"Unsupported Python version {platform.python_version()}. "
        "Railway Headway Simulator requires Python 3.10 or newer."
    )
print("✔ Runtime environment compatibility confirmed.")


=== Google Colab Runtime Verification ===
Python Version: 3.13.16
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
✔ Runtime environment compatibility confirmed.


In [17]:
# Step 2: Install Package and Dependencies
import os
import sys
import subprocess
from pathlib import Path

# Lütfen buraya kendi GitHub depo adresinizi yazın (örneğin: ay-simulator.git)
GIT_REPO_URL = "https://github.com/ucwebchess/railway-headway-simulator"

REPO_NAME = "railway-headway-simulator"
REPO_DIR = Path.cwd() / REPO_NAME

# Automatically attempt to clone if not present
if not REPO_DIR.exists():
    print(f"'{REPO_NAME}' klasörü bulunamadı. '{GIT_REPO_URL}' adresinden klonlanmaya çalışılıyor...")
    clone_result = subprocess.run(["git", "clone", GIT_REPO_URL], capture_output=True, text=True)

    if clone_result.returncode != 0:
        print("\nHATA: Klonlama başarısız oldu!")
        print("Lütfen şunlardan birini yapın:")
        print("1. Kodun başındaki 'GIT_REPO_URL' değişkenine geçerli git adresinizi yazın.")
        print("2. Veya proje klasörünü (railway-headway-simulator) sol menüden Colab'e sürükleyip yükleyin.\n")
        REPO_DIR = Path.cwd()
    else:
        print("✔ Depo başarıyla klonlandı.")

print(f"Çalışma Dizini: {REPO_DIR}")

# Install package in editable/local mode
print("Paket kuruluyor...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "."],
    capture_output=True,
    text=True,
    cwd=REPO_DIR
)

if result.returncode != 0:
    print("Kurulum başarısız oldu. Çıktı:")
    print(result.stderr)
    raise RuntimeError("Paket yüklenemedi. Lütfen doğru dizinde olduğunuzdan ve pyproject.toml dosyasının varlığından emin olun.")

print("✔ Paket başarıyla kuruldu.")

'railway-headway-simulator' klasörü bulunamadı. 'https://github.com/ucwebchess/railway-headway-simulator' adresinden klonlanmaya çalışılıyor...
✔ Depo başarıyla klonlandı.
Çalışma Dizini: /content/railway-headway-simulator
Paket kuruluyor...
✔ Paket başarıyla kuruldu.


In [20]:
# Step 3: Initialize Configuration, Logging, and Directories
import sys
from pathlib import Path

# Add the cloned repository root and its possible 'src' directory to Python's path
repo_path = Path.cwd() / "railway-headway-simulator"
src_path = repo_path / "src"

if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from headway.core.configuration import AppConfig, EnvironmentType
from headway.core.logging_config import setup_logging, get_logger
from headway.core.version import __version__, get_build_info

# Initialize Colab-optimized configuration
config = AppConfig(
    environment=EnvironmentType.COLAB,
    log_level="INFO",
    dev_mode=True
)
config.initialize_runtime()

# Initialize centralized logging
setup_logging(
    level=config.log_level,
    log_dir=config.directories.logs_dir
)

logger = get_logger("colab.launcher")
logger.info("Initializing Railway Headway Simulator v%s for Colab", __version__)
print(f"✔ Runtime initialized. Build info: {get_build_info()}")
print(f"✔ Storage directories ready at: {config.directories.base_dir}")

2026-10-09 21:00:10 [INFO] headway.colab.launcher: Initializing Railway Headway Simulator v0.1.0-dev for Colab
✔ Runtime initialized. Build info: {'app_name': 'Railway Headway & Capacity Simulator', 'version': '0.1.0-dev', 'version_info': (0, 1, 0, 'dev'), 'phase': 'P00 — Project Foundation', 'status': 'Development'}
✔ Storage directories ready at: /content/storage


In [21]:
# Step 4: Launch Gradio Interface
import gradio as gr
from headway.ui.app import create_app

# Close any existing active Gradio servers to prevent port collisions on re-run
try:
    gr.close_all()
except Exception:
    pass

# Create the Gradio interface
demo = create_app(config)

# Launch with inline Colab display
# Note: share=False by default to protect proprietary engineering datasets
demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=False,
    inline=True,
    show_error=True,
    theme=getattr(demo, "app_theme", None),
    css=getattr(demo, "app_css", None),
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

### Step 5: Troubleshooting & Operation Notes

- **Restarting the UI:** If you need to reload the UI after modifying configuration, re-run Cell 4 and Cell 5. The launcher automatically executes `gr.close_all()` to release port 7860.
- **Port Conflict:** If port 7860 is occupied by another process, change `server_port=7861` in `config.server`.
- **Public Share Link:** If remote browser access outside Colab is strictly needed and permitted, set `config.server.share = True` before launching.
- **Runtime Persistence:** Files stored in Google Colab's local filesystem are temporary and destroyed when the runtime disconnects. Ensure all project packages and exports are downloaded to your local computer.
